## Goal: Finalize analysis to answer our 3 supporting questions
- Do fires cross wetland boundaries? (yes, supported by fact that there is wetland area burned each year)
    - 
- Are wildfires less frequent near wetlands? 
    - Goal: use chi squared test to see if burned area inside wetlands is about what you would expect for average of all burned area or if it is statistically different

In [1]:
# import packages

import platform
from pathlib import Path
import pandas as pd
import geopandas as gpd
import seaborn as sns
import calendar
import matplotlib.pyplot as plt
import pygris
from shapely.geometry import Point
import numpy as np


In [13]:
# import data 

# filepath

project_fldr = Path.cwd().parents[1]
raw_fldr = project_fldr / 'data' / 'raw'
processed_fldr = project_fldr / 'data' / 'processed'

# #  fire data by year
fire_2020 = gpd.read_file(processed_fldr / "fire_explore/fire_2020.geojson")
fire_2021 = gpd.read_file(processed_fldr / "fire_explore/fire_2021.geojson")
fire_2022 = gpd.read_file(processed_fldr / "fire_explore/fire_2022.geojson")
fire_2023 = gpd.read_file(processed_fldr / "fire_explore/fire_2023.geojson")
fire_2024 = gpd.read_file(processed_fldr / "fire_explore/fire_2024.geojson")
fire_2025 = gpd.read_file(processed_fldr / "fire_explore/fire_2025.geojson")

# california outline
ca_outline = gpd.read_file(processed_fldr /"fire_explore/ca_outline.geojson")


# set a path to the main box folder based on machine's os
# if windows: 
if platform.system() == 'Windows':
    box_path = Path.home() / 'Box'
# if mac:
else:
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

#Confirm that the box path is valid
if box_path.exists():
    print(f'Box folder found at {box_path}')
else:
    print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

# create path object that goes to the specific folder on Box drive
wetlands_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'wetlands_subset' / 'wetlands_subset.geojson'
print(wetlands_path.exists())

# read in the wetlands geojson file, this will take a WHILE - 10 mins 1st try 
wetlands = gpd.read_file(wetlands_path, engine="pyogrio")



Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True


In [14]:
# convert CRS function
def crs_convert(shp):
    shp = shp.to_crs(epsg=26910)
    return shp

# check CRS of files
print(fire_2020.crs)
print(ca_outline.crs)

EPSG:4326
EPSG:4326


In [15]:
# convert all shp CRS
fire_2020 = crs_convert(fire_2020)
fire_2021 = crs_convert(fire_2021)
fire_2022 = crs_convert(fire_2022)
fire_2023 = crs_convert(fire_2023)
fire_2024 = crs_convert(fire_2024)
fire_2025 = crs_convert(fire_2025)
ca_outline = crs_convert(ca_outline)
wetlands = crs_convert(wetlands)

In [18]:
print(gpd.GeoSeries.sample_points.__doc__)



        Sample points from each geometry.

        Generate a MultiPoint per each geometry containing points sampled from the
        geometry. You can either sample randomly from a uniform distribution or use an
        advanced sampling algorithm from the ``pointpats`` package.

        For polygons, this samples within the area of the polygon. For lines,
        this samples along the length of the linestring. For multi-part
        geometries, the weights of each part are selected according to their relevant
        attribute (area for Polygons, length for LineStrings), and then points are
        sampled from each part.

        Any other geometry type (e.g. Point, GeometryCollection) is ignored, and an
        empty MultiPoint geometry is returned.

        Parameters
        ----------
        size : int | array-like
            The size of the sample requested. Indicates the number of samples to draw
            from each geometry.  If an array of the same length as a GeoSerie

In [ ]:
# Generate 10,000 random points
n_pts = 10000

# Extract California polygon
study_area = ca_outline.geometry.iloc[0]

# Generate random points
random_pts = gpd.GeoSeries(
    [study_area],
    crs=ca_outline.crs
).sample_points( # take a random sample
    size=n_pts,
    method="uniform",
    seed=42 # but get the same random pts every time i run this code
).explode(index_parts=False).reset_index(drop=True) # separate each pt into its own line

# Convert to GeoDataFrame
random_pts = gpd.GeoDataFrame(
    geometry=random_pts,
    crs=ca_outline.crs
)


In [20]:
# place fires in dict for easy access
fire_dict = {
    2020: fire_2020,
    2021: fire_2021,
    2022: fire_2022,
    2023: fire_2023,
    2024: fire_2024,
    2025: fire_2025
}

In [ ]:
# function for chi squared analysis
def classify_pts(fire):
    # create copy of random points
    points = random_pts.copy().reset_index(drop=True)

    # create a copy of wetlands
    wetlands_cpy = wetlands.copy().reset_index(drop=True)
    
    # identify wetland intersections
    wetland_join = gpd.sjoin(
        points, 
        wetlands,
        how='left'
    )
    
    # classify wetland presence (1=Y, 0=N)
    points['wetland'] = ( # add column called "wetland" to points 
        wetland_join['index_right'] # creates column that shows corresponding wetland polygon to point if any
        .notna() # creates T/F column where T = intersects (has a number)
        .groupby(level=0) # combines any points that intersect with multiple wetland shapefiles
        .any() # if a pt overlaps with at least 1 wetland, it is T
        .astype(int) # changes T/F boolean to 1/0 values 

    ) 

    # identify fire intersections














